# NB-07: RQ3 — Country Archetypes (K-Means Clustering with Bootstrap Stability)

**The Peacekeepers' Arms Race — Stability-Instability Paradox**

Tests whether countries cluster into distinct conflict-capability archetypes — provisionally labelled
Safe Hegemons, Armed Instabilizers, and Defensive Deterrents — and whether those clusters remain
stable under bootstrap resampling and out-of-sample validation.

**Five-phase analysis:**
1. Feature engineering — country-level means from 1989–2024 panel
2. k selection — silhouette score + gap statistic for k = 2 to 8
3. Bootstrap stability — 100 iterations, Jaccard similarity with Hungarian alignment
4. Out-of-sample validation — train on pre-2019 features, ANOVA on post-2019 conflict
5. Centroid inspection and archetype labelling

**Primary MTS:** `mts_pca_3feat`. **Robustness:** `mts_milex`, `mts_tiv`.

**Cluster stability threshold:** mean Jaccard > 0.75 across bootstrap iterations.

## Section 0 — Setup

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import f_oneway
from scipy.optimize import linear_sum_assignment
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

from src.io_utils import load_checkpoint, save_checkpoint
from src.config import CLEAN_DIR, FIGURES_DIR, TABLES_DIR, SEED

np.random.seed(SEED)

plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
sns.set_palette("tab10")

FIG_DIR = FIGURES_DIR / "nb07"
TBL_DIR = TABLES_DIR  / "nb07"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR.mkdir(parents=True, exist_ok=True)

# ── Load inputs ─────────────────────────────────────────────────────────────
rq3    = load_checkpoint(CLEAN_DIR / "rq3_cross_section.parquet")
master = load_checkpoint(CLEAN_DIR / "master_panel.parquet")
rq1    = load_checkpoint(CLEAN_DIR / "rq1_panel.parquet")

print(f"rq3_cross_section: {rq3.shape}")
print(f"master_panel:      {master.shape}")
print(f"rq1_panel:         {rq1.shape}")
print(f"\nrq3 columns: {list(rq3.columns)}")

# ── Global configuration ─────────────────────────────────────────────────────
PRIMARY_MTS  = "mts_pca_3feat"
K_RANGE      = range(2, 9)           # test k = 2..8
N_BOOTSTRAP  = 100
JACCARD_THRESHOLD = 0.75             # stability criterion
OOS_CUTOFF   = 2019                  # train on <2019, validate on 2019+
ANALYSIS_MIN = 1989
ANALYSIS_MAX = 2024

CONFLICT_COLS = [
    "part_n_war", "part_n_minor", "war_minor_ratio",
    "part_n_extraterritorial", "extraterritorial_share",
]
MTS_COLS     = ["mts_pca_3feat", "mts_milex", "mts_tiv"]
CONTROL_COLS = ["vdem_v2x_polyarchy"]

HEADLINE_COUNTRIES = [
    "USA", "RUS", "CHN", "IND", "GBR", "FRA", "SAU", "PAK",
    "ISR", "TUR", "IRN", "NGA", "BRA", "ETH", "SDN", "COD",
    "SYR", "IRQ", "AFG", "UKR",
]

print(f"\nPrimary MTS:          {PRIMARY_MTS}")
print(f"k range:              {list(K_RANGE)}")
print(f"Bootstrap iterations: {N_BOOTSTRAP}")
print(f"Jaccard threshold:    {JACCARD_THRESHOLD}")
print(f"OOS cutoff:           {OOS_CUTOFF}")
print("\nSection 0 complete.")

[checkpoint] loaded ← rq3_cross_section.parquet  (192 rows)
[checkpoint] loaded ← master_panel.parquet  (15,168 rows)
[checkpoint] loaded ← rq1_panel.parquet  (6,912 rows)
rq3_cross_section: (192, 14)
master_panel:      (15168, 61)
rq1_panel:         (6912, 23)

rq3 columns: ['iso3', 'mean_milex', 'mean_tiv', 'total_conflicts', 'mean_war_minor_ratio', 'mean_extraterritorial_share', 'mean_polyarchy', 'mean_gdp_per_capita', 'casualty_intensity', 'region', 'mts_milex', 'mts_tiv', 'mts_pca', 'mts_pca_3feat']

Primary MTS:          mts_pca_3feat
k range:              [2, 3, 4, 5, 6, 7, 8]
Bootstrap iterations: 100
Jaccard threshold:    0.75
OOS cutoff:           2019

Section 0 complete.


## Section 1 — Feature Engineering

Country-level averages over the 1989–2024 window form the clustering input. The eight features are chosen
to span capability, conflict composition, force-projection, regime type, and general conflict propensity.
`mts_tiv` is excluded from the primary feature set due to high structural missingness (~40% of country-years
have no TIV imports); it is retained as a robustness check in Section 5.

Missing values in `war_minor_ratio` and `extraterritorial_share` — which are undefined for countries with
zero conflict — are imputed to the column median before standardization. This is conservative: it prevents
all-peaceful countries from being forced into extreme positions on ratio-based features.

Features used:
- `mts_pca_3feat_mean` — primary capability level (PCA 3-feature spec, 1989–2024)
- `mts_milex_mean` — capability — defence spending dimension
- `part_n_war_mean` — mean annual interstate war participations
- `part_n_minor_mean` — mean annual minor conflict participations
- `war_minor_ratio_mean` — mean compositional shift indicator
- `extraterritorial_share_mean` — mean proportion of conflicts fought abroad
- `vdem_v2x_polyarchy_mean` — mean democracy/regime score
- `conflict_active_pct` — proportion of years with any conflict participation

In [ ]:
# ── Aggregate rq1_panel to country-level means ────────────────────────────
rq1_sub = rq1[
    (rq1["year"] >= ANALYSIS_MIN) & (rq1["year"] <= ANALYSIS_MAX)
].copy()

agg_cols = [c for c in CONFLICT_COLS + MTS_COLS + CONTROL_COLS if c in rq1_sub.columns]
agg_dict = {c: "mean" for c in agg_cols}

country_features = (
    rq1_sub.groupby("iso3")
    .agg(agg_dict)
    .reset_index()
    .rename(columns={c: f"{c}_mean" for c in agg_cols})
)

# ── Conflict-activity rate: % years with any participation ────────────────
activity = (
    rq1_sub.assign(
        any_conflict=lambda d: (
            (d["part_n_war"].fillna(0) + d["part_n_minor"].fillna(0)) > 0
        ).astype(float)
    )
    .groupby("iso3")["any_conflict"]
    .mean()
    .rename("conflict_active_pct")
    .reset_index()
)
country_features = country_features.merge(activity, on="iso3", how="left")

print(f"Country-level features: {country_features.shape}")
print(f"\nMissingness per column (top 10):")
print(country_features.isnull().sum().sort_values(ascending=False).head(12).to_string())

# ── Define clustering feature set ────────────────────────────────────────
CLUSTER_FEATURES = [
    "mts_pca_3feat_mean",
    "mts_milex_mean",
    "part_n_war_mean",
    "part_n_minor_mean",
    "war_minor_ratio_mean",
    "extraterritorial_share_mean",
    "vdem_v2x_polyarchy_mean",
    "conflict_active_pct",
]

# ── BRD casualty intensity feature (conditional on data availability) ─────────
brd_candidates_master = [c for c in master.columns
                          if any(x in c.lower() for x in ["brd", "death", "casualties"])]
BRD_COL = None
for cand in ["brd_best_mean", "brd_best", "deaths_best", "brd_mean"]:
    if cand in master.columns or cand in rq1.columns:
        BRD_COL = cand
        break
if BRD_COL is None and brd_candidates_master:
    BRD_COL = brd_candidates_master[0]

if BRD_COL is not None:
    src_df = rq1_sub if BRD_COL in rq1_sub.columns else (
        master[(master["year"] >= ANALYSIS_MIN) & (master["year"] <= ANALYSIS_MAX)]
        if BRD_COL in master.columns else None
    )
    if src_df is not None:
        brd_agg = (
            src_df.groupby("iso3")[BRD_COL]
            .mean()
            .rename("log_brd_mean")
            .apply(np.log1p)
            .reset_index()
        )
        country_features = country_features.merge(brd_agg, on="iso3", how="left")
        n_nonzero_brd = (country_features["log_brd_mean"] > 0).sum()
        if n_nonzero_brd >= 30:
            CLUSTER_FEATURES = CLUSTER_FEATURES + ["log_brd_mean"]
            print(f"BRD feature added to clustering: log_brd_mean ({n_nonzero_brd} non-zero countries)")
        else:
            print(f"BRD coverage too low for clustering ({n_nonzero_brd} countries) — skipping.")
else:
    print("No BRD column found in master_panel or rq1_panel — skipping intensity feature.")
    print(f"  Searched: {brd_candidates_master[:5]}")

# Require primary MTS and at least one conflict column
feat_df = country_features.dropna(
    subset=["mts_pca_3feat_mean", "part_n_war_mean", "part_n_minor_mean"]
).copy()

# Impute remaining missing values to column median (ratio features undefined for
# countries with zero conflict; imputing to median rather than 0 avoids artefacts)
feat_cols = [c for c in CLUSTER_FEATURES if c in feat_df.columns]
for col in feat_cols:
    n_missing = feat_df[col].isna().sum()
    if n_missing > 0:
        med = feat_df[col].median()
        feat_df[col] = feat_df[col].fillna(med)
        print(f"  Imputed {n_missing} NaN in {col} → median {med:.4f}")

feat_df = feat_df[["iso3"] + feat_cols].reset_index(drop=True)

print(f"\nClean feature matrix: {feat_df.shape} ({len(feat_cols)} features × {len(feat_df)} countries)")
print(f"\nFeature summary:")
print(feat_df[feat_cols].describe().round(3).to_string())

# ── Standardize ─────────────────────────────────────────────────────────
scaler = StandardScaler()
X = scaler.fit_transform(feat_df[feat_cols].values)
country_labels_arr = feat_df["iso3"].values

print(f"\nX (standardized): {X.shape}")

feat_df.to_csv(TBL_DIR / "section1_feature_matrix.csv", index=False)
print(f"Saved → {TBL_DIR / 'section1_feature_matrix.csv'}")
print("\nSection 1 complete.")

## Section 2 — k Selection

Two complementary criteria determine the optimal number of clusters:

**Silhouette score** measures how well each country fits its assigned cluster versus the next-nearest
cluster. Values near +1 indicate tight, well-separated clusters. The optimal k is the peak.

**Gap statistic** (Tibshirani et al., 2001) compares the within-cluster dispersion of the real data
to that of random uniform reference data. The Tibshirani criterion selects the *smallest* k where
$\text{gap}(k) \geq \text{gap}(k+1) - s_{k+1}$, where $s_{k+1}$ is the simulation-adjusted standard error.

Both criteria are reported. If they agree, that k is used. If they disagree, silhouette is preferred as
the more robust criterion for non-uniform distributions typical in IR country panels.

In [3]:
# ── Gap statistic implementation ──────────────────────────────────────────
def gap_statistic(X_scaled, k_max, n_ref=50, seed=SEED):
    """
    Tibshirani et al. (2001) gap statistic.
    Returns gaps[0..k_max-1], sks[0..k_max-1], log_W_real[0..k_max-1].
    Optimal k (Tibshirani criterion): smallest k where gap(k) >= gap(k+1) - sk(k+1).
    """
    rng = np.random.default_rng(seed)
    log_W_real   = []
    log_W_ref_by_k = []

    for k in range(1, k_max + 1):
        km = KMeans(n_clusters=k, random_state=seed, n_init=10, max_iter=300)
        km.fit(X_scaled)
        log_W_real.append(np.log(km.inertia_ + 1e-10))

        ref_logWs = []
        for _ in range(n_ref):
            X_ref = rng.uniform(
                X_scaled.min(axis=0), X_scaled.max(axis=0), size=X_scaled.shape
            )
            km_ref = KMeans(n_clusters=k, random_state=seed, n_init=10, max_iter=300)
            km_ref.fit(X_ref)
            ref_logWs.append(np.log(km_ref.inertia_ + 1e-10))
        log_W_ref_by_k.append(ref_logWs)

    gaps = np.array([
        np.mean(log_W_ref_by_k[k - 1]) - log_W_real[k - 1]
        for k in range(1, k_max + 1)
    ])
    sks = np.array([
        np.std(log_W_ref_by_k[k - 1]) * np.sqrt(1.0 + 1.0 / n_ref)
        for k in range(1, k_max + 1)
    ])
    return gaps, sks, np.array(log_W_real)


# ── Silhouette scores ──────────────────────────────────────────────────────
print("Computing silhouette scores (k = 2..8) ...")
sil_scores = {}
inertias   = {}
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=20, max_iter=300)
    labels = km.fit_predict(X)
    sil_scores[k] = silhouette_score(X, labels)
    inertias[k]   = km.inertia_
    print(f"  k={k}: silhouette={sil_scores[k]:.4f}  inertia={inertias[k]:.1f}")

# ── Gap statistic ──────────────────────────────────────────────────────────
print("\nComputing gap statistic (50 reference datasets per k) ...")
gaps, sks, log_W = gap_statistic(X, k_max=max(K_RANGE), n_ref=50, seed=SEED)
print("  Done.")

# Tibshirani criterion
k_arr = np.arange(1, max(K_RANGE) + 1)
gap_optimal_k = None
for i, k in enumerate(k_arr[:-1]):
    if k >= min(K_RANGE) and gaps[i] >= gaps[i + 1] - sks[i + 1]:
        gap_optimal_k = int(k)
        break
if gap_optimal_k is None:
    gap_optimal_k = int(k_arr[np.argmax(gaps)])

sil_optimal_k = max(sil_scores, key=sil_scores.get)

print(f"\nSilhouette-optimal k : {sil_optimal_k}  (score = {sil_scores[sil_optimal_k]:.4f})")
print(f"Gap-optimal k        : {gap_optimal_k}")

if sil_optimal_k == gap_optimal_k:
    OPTIMAL_K = sil_optimal_k
    print(f"\nBoth criteria agree → OPTIMAL_K = {OPTIMAL_K}")
else:
    OPTIMAL_K = sil_optimal_k
    print(f"\nCriteria disagree — using silhouette → OPTIMAL_K = {OPTIMAL_K}")
    print(f"  (Gap suggests k={gap_optimal_k}; retained in table for transparency.)")

# ── Save k-selection table ────────────────────────────────────────────────
k_sel_df = pd.DataFrame({
    "k":          list(K_RANGE),
    "silhouette": [sil_scores[k] for k in K_RANGE],
    "inertia":    [inertias[k]   for k in K_RANGE],
    "gap":        [gaps[k - 1]   for k in K_RANGE],
    "gap_sk":     [sks[k - 1]    for k in K_RANGE],
})
k_sel_df.to_csv(TBL_DIR / "section2_k_selection.csv", index=False)
print(f"\nSaved → {TBL_DIR / 'section2_k_selection.csv'}")
print("\nSection 2 complete.")

Computing silhouette scores (k = 2..8) ...
  k=2: silhouette=0.2491  inertia=918.6
  k=3: silhouette=0.2572  inertia=735.5
  k=4: silhouette=0.2530  inertia=640.5
  k=5: silhouette=0.2183  inertia=576.6
  k=6: silhouette=0.2270  inertia=516.6
  k=7: silhouette=0.2425  inertia=461.4
  k=8: silhouette=0.2407  inertia=431.4

Computing gap statistic (50 reference datasets per k) ...
  Done.

Silhouette-optimal k : 3  (score = 0.2572)
Gap-optimal k        : 4

Criteria disagree — using silhouette → OPTIMAL_K = 3
  (Gap suggests k=4; retained in table for transparency.)

Saved → D:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\tables\nb07\section2_k_selection.csv

Section 2 complete.


### Section 2b — Ward Hierarchical Clustering Robustness

Ward linkage minimises total within-cluster variance at each merge step and makes no
spherical-cluster assumption, unlike K-Means. Running both algorithms provides a robustness
check: if Ward and K-Means agree on k and on cluster membership for headline countries,
the partition is not an artefact of the K-Means algorithm.

A dendrogram truncated to the top 15 merges is plotted to visualise the hierarchical
structure. The optimal Ward cut is the level with the largest inter-merge distance gap
("elbow" in the dendrogram height).

In [ ]:
# Ensure reference K-Means labels are available for Ward alignment comparison
km_s2b_ref = KMeans(n_clusters=OPTIMAL_K, random_state=SEED, n_init=20, max_iter=300)
final_labels = km_s2b_ref.fit_predict(X)

from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform

# ── Ward linkage silhouette (same k range) ─────────────────────────────────
print("Computing Ward silhouette scores ...")
ward_sil = {}
for k in K_RANGE:
    ward = AgglomerativeClustering(n_clusters=k, linkage="ward")
    ward_labels = ward.fit_predict(X)
    ward_sil[k] = silhouette_score(X, ward_labels)
    print(f"  k={k}: Ward silhouette = {ward_sil[k]:.4f}  |  K-Means = {sil_scores[k]:.4f}")

ward_optimal_k = max(ward_sil, key=ward_sil.get)
print(f"\nWard-optimal k:    {ward_optimal_k}  (silhouette = {ward_sil[ward_optimal_k]:.4f})")
print(f"K-Means-optimal k: {sil_optimal_k}  (silhouette = {sil_scores[sil_optimal_k]:.4f})")

if ward_optimal_k == OPTIMAL_K:
    print(f"\n✓ Ward and K-Means agree on k={OPTIMAL_K} — partition is algorithm-robust.")
else:
    print(f"\n! Ward and K-Means disagree (Ward={ward_optimal_k}, K-Means={OPTIMAL_K}).")
    print(f"  Using K-Means optimal k={OPTIMAL_K} as primary; Ward={ward_optimal_k} reported.")

# ── Headline-country cluster agreement ────────────────────────────────────
ward_final = AgglomerativeClustering(n_clusters=OPTIMAL_K, linkage="ward")
ward_labels_final = ward_final.fit_predict(X)

# Align Ward cluster IDs to K-Means IDs by majority vote per cluster
from scipy.optimize import linear_sum_assignment as lsa
km_sets   = [set(np.where(final_labels == c)[0])  for c in range(OPTIMAL_K)]
ward_sets = [set(np.where(ward_labels_final == c)[0]) for c in range(OPTIMAL_K)]
overlap   = np.zeros((OPTIMAL_K, OPTIMAL_K))
for r in range(OPTIMAL_K):
    for c in range(OPTIMAL_K):
        overlap[r, c] = len(km_sets[r] & ward_sets[c])
row_ind, col_ind = lsa(-overlap)
ward_to_km = {col_ind[i]: row_ind[i] for i in range(OPTIMAL_K)}
ward_labels_aligned = np.array([ward_to_km[l] for l in ward_labels_final])

agreement_rate = (final_labels == ward_labels_aligned).mean()
print(f"\nOverall country assignment agreement (K-Means vs Ward): {agreement_rate:.1%}")

disagreements = feat_df[final_labels != ward_labels_aligned].copy()
disagreements["km_cluster"]   = final_labels[final_labels != ward_labels_aligned]
disagreements["ward_cluster"]  = ward_labels_aligned[final_labels != ward_labels_aligned]
print(f"Countries assigned differently: {len(disagreements)}")
if len(disagreements) > 0:
    hl_disagree = disagreements[disagreements["iso3"].isin(HEADLINE_COUNTRIES)]
    if not hl_disagree.empty:
        print("Headline country disagreements:")
        print(hl_disagree[["iso3", "km_cluster", "ward_cluster"]].to_string(index=False))

# ── Dendrogram (top 15 merges) ────────────────────────────────────────────
Z_linkage = linkage(X, method="ward")
fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(
    Z_linkage, ax=ax,
    truncate_mode="lastp", p=15,
    leaf_rotation=45, leaf_font_size=9,
    show_contracted=True,
    color_threshold=Z_linkage[-OPTIMAL_K + 1, 2],
)
ax.set_title(f"Ward Dendrogram (top 15 merges, cut at k={OPTIMAL_K})", fontsize=11)
ax.set_xlabel("Country cluster (size in parentheses)")
ax.set_ylabel("Merge distance")
ax.axhline(Z_linkage[-OPTIMAL_K + 1, 2], color="firebrick",
           linestyle="--", lw=1.2, label=f"Cut at k={OPTIMAL_K}")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1b_ward_dendrogram.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"\nDendrogram saved → {FIG_DIR / 'fig1b_ward_dendrogram.png'}")

# Save Ward comparison table
ward_sil_df = pd.DataFrame({
    "k":                 list(K_RANGE),
    "ward_silhouette":   [ward_sil[k]    for k in K_RANGE],
    "kmeans_silhouette": [sil_scores[k]  for k in K_RANGE],
})
ward_sil_df.to_csv(TBL_DIR / "section2b_ward_vs_kmeans.csv", index=False)
print(f"Saved → {TBL_DIR / 'section2b_ward_vs_kmeans.csv'}")

## Section 3 — Bootstrap Stability

Cluster stability is measured by resampling countries with replacement 100 times and comparing each
bootstrap clustering to the reference (full-sample) clustering. Alignment between bootstrap and reference
clusters uses the **Hungarian algorithm** (optimal bipartite matching on a Jaccard cost matrix), which
avoids the label-permutation problem in unsupervised evaluation.

**Jaccard similarity** $J(A, B) = |A \cap B| / |A \cup B|$ compares two sets of country indices.
A cluster with mean Jaccard > 0.75 across bootstrap iterations is considered stable.

Unstable clusters (J ≤ 0.75) are reported as such in the archetype table; their centroid profiles and
headline-country membership are still shown but clearly flagged.

In [4]:
def jaccard_similarity(set_a, set_b):
    a, b = set(set_a), set(set_b)
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)


def bootstrap_cluster_stability(X_scaled, k, n_boot=N_BOOTSTRAP, seed=SEED):
    """
    Bootstrap Jaccard stability with Hungarian alignment.

    Returns
    -------
    mean_jacc : np.ndarray, shape (k,)
        Mean Jaccard per reference cluster across n_boot iterations.
    all_jacc  : np.ndarray, shape (n_boot, k)
        Per-iteration Jaccard matrix (NaN where alignment failed).
    ref_labels : np.ndarray, shape (n_countries,)
        Reference cluster assignments from the full dataset.
    """
    rng = np.random.default_rng(seed)
    n   = len(X_scaled)

    km_ref    = KMeans(n_clusters=k, random_state=seed, n_init=20, max_iter=300)
    ref_labels = km_ref.fit_predict(X_scaled)
    ref_sets   = [set(np.where(ref_labels == c)[0]) for c in range(k)]

    all_jacc = np.full((n_boot, k), np.nan)

    for b in range(n_boot):
        idx    = rng.choice(n, size=n, replace=True)
        X_boot = X_scaled[idx]
        km_b   = KMeans(n_clusters=k, random_state=seed + b + 1, n_init=10, max_iter=300)
        boot_labels = km_b.fit_predict(X_boot)

        # Map bootstrap cluster labels back to original-index space
        boot_sets = [set(idx[boot_labels == c]) for c in range(k)]

        # Build k×k Jaccard cost matrix; negate for minimization
        cost = np.zeros((k, k))
        for r in range(k):
            for c in range(k):
                cost[r, c] = -jaccard_similarity(ref_sets[r], boot_sets[c])

        row_ind, col_ind = linear_sum_assignment(cost)
        for r, c in zip(row_ind, col_ind):
            all_jacc[b, r] = -cost[r, c]

    return np.nanmean(all_jacc, axis=0), all_jacc, ref_labels


print(f"Running bootstrap stability (k={OPTIMAL_K}, n_boot={N_BOOTSTRAP}) ...")
mean_jacc, all_jacc, final_labels = bootstrap_cluster_stability(
    X, OPTIMAL_K, n_boot=N_BOOTSTRAP, seed=SEED
)

# Refit final KMeans separately to ensure deterministic reference labelling
km_final    = KMeans(n_clusters=OPTIMAL_K, random_state=SEED, n_init=20, max_iter=300)
final_labels = km_final.fit_predict(X)

stable_clusters = [int(c) for c in range(OPTIMAL_K) if mean_jacc[c] >= JACCARD_THRESHOLD]

print(f"\nBootstrap Jaccard per cluster:")
for c in range(OPTIMAL_K):
    flag = "STABLE" if mean_jacc[c] >= JACCARD_THRESHOLD else "UNSTABLE"
    print(f"  Cluster {c}: mean Jaccard = {mean_jacc[c]:.3f}  std = {np.nanstd(all_jacc[:, c]):.3f}  [{flag}]")

print(f"\nStable clusters (J > {JACCARD_THRESHOLD}): {stable_clusters}")

# ── Save ─────────────────────────────────────────────────────────────────
boot_iter_df = pd.DataFrame(
    all_jacc, columns=[f"cluster_{c}" for c in range(OPTIMAL_K)]
)
boot_iter_df.insert(0, "iteration", range(N_BOOTSTRAP))
boot_iter_df.to_csv(TBL_DIR / "section3_bootstrap_iterations.csv", index=False)

boot_summary = pd.DataFrame({
    "cluster":      list(range(OPTIMAL_K)),
    "mean_jaccard": mean_jacc.round(4),
    "std_jaccard":  np.nanstd(all_jacc, axis=0).round(4),
    "stable":       [j >= JACCARD_THRESHOLD for j in mean_jacc],
})
boot_summary.to_csv(TBL_DIR / "section3_bootstrap_summary.csv", index=False)
print(f"Saved → {TBL_DIR}")
print("\nSection 3 complete.")

Running bootstrap stability (k=3, n_boot=100) ...

Bootstrap Jaccard per cluster:
  Cluster 0: mean Jaccard = 0.549  std = 0.096  [UNSTABLE]
  Cluster 1: mean Jaccard = 0.533  std = 0.113  [UNSTABLE]
  Cluster 2: mean Jaccard = 0.569  std = 0.069  [UNSTABLE]

Stable clusters (J > 0.75): []
Saved → D:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\tables\nb07

Section 3 complete.


## Section 4 — Out-of-Sample Validation

Clustering is re-estimated on features built from **1989–2018 data only**. The resulting cluster
assignments are then tested against **2019–2024 conflict outcomes** via a one-way ANOVA.

The validation question is: does cluster membership (derived entirely from pre-2019 information)
predict the level of conflict a country experiences in the following six years?

- If ANOVA is significant (p < 0.05): the clusters are real archetypes with predictive content —
  not just a taxonomic partition of known-in-sample variation.
- If ANOVA is not significant: the clusters may still be descriptively valid but do not generalise
  forward; this is an important limitation to report.

Note: the scaler is re-fitted on the training features only, to prevent data leakage from
the post-2019 period.

In [5]:
TRAIN_MAX = OOS_CUTOFF - 1   # 2018
TEST_MIN  = OOS_CUTOFF        # 2019

# ── Build pre-2019 feature matrix (same pipeline as Section 1) ────────────
rq1_train = rq1[
    (rq1["year"] >= ANALYSIS_MIN) & (rq1["year"] <= TRAIN_MAX)
].copy()

agg_cols_tr = [c for c in CONFLICT_COLS + MTS_COLS + CONTROL_COLS if c in rq1_train.columns]
feat_train  = (
    rq1_train.groupby("iso3")
    .agg({c: "mean" for c in agg_cols_tr})
    .reset_index()
    .rename(columns={c: f"{c}_mean" for c in agg_cols_tr})
)

activity_tr = (
    rq1_train.assign(
        any_conflict=lambda d: (
            (d["part_n_war"].fillna(0) + d["part_n_minor"].fillna(0)) > 0
        ).astype(float)
    )
    .groupby("iso3")["any_conflict"]
    .mean()
    .rename("conflict_active_pct")
    .reset_index()
)
feat_train = feat_train.merge(activity_tr, on="iso3", how="left")
feat_train = feat_train.dropna(subset=["mts_pca_3feat_mean", "part_n_war_mean"])

feat_cols_tr = [c for c in feat_cols if c in feat_train.columns]
for col in feat_cols_tr:
    feat_train[col] = feat_train[col].fillna(feat_train[col].median())

feat_train = feat_train[["iso3"] + feat_cols_tr].reset_index(drop=True)

# Re-fit scaler on training features only (no leakage)
scaler_tr = StandardScaler()
X_train   = scaler_tr.fit_transform(feat_train[feat_cols_tr].values)

print(f"Training feature matrix (1989–{TRAIN_MAX}): {feat_train.shape}")

# ── Fit clustering on pre-2019 ────────────────────────────────────────────
km_oos   = KMeans(n_clusters=OPTIMAL_K, random_state=SEED, n_init=20, max_iter=300)
oos_labels = km_oos.fit_predict(X_train)
feat_train = feat_train.copy()
feat_train["cluster_oos"] = oos_labels

# ── Post-2019 conflict intensity ──────────────────────────────────────────
rq1_test = rq1[rq1["year"] >= TEST_MIN].copy()
post_conflict = (
    rq1_test.groupby("iso3")[["part_n_war", "part_n_minor"]]
    .mean()
    .assign(total_conflict=lambda d: d["part_n_war"].fillna(0) + d["part_n_minor"].fillna(0))
    .reset_index()
)

oos_val = feat_train[["iso3", "cluster_oos"]].merge(
    post_conflict[["iso3", "total_conflict"]], on="iso3", how="inner"
)

print(f"OOS validation countries: {len(oos_val)}")
print(f"Year range (test):        {TEST_MIN}–{ANALYSIS_MAX}")

# ── One-way ANOVA ──────────────────────────────────────────────────────────
groups = [
    oos_val.loc[oos_val["cluster_oos"] == c, "total_conflict"].dropna().values
    for c in range(OPTIMAL_K)
]
groups_valid = [g for g in groups if len(g) >= 3]

if len(groups_valid) >= 2:
    F_stat, p_val = f_oneway(*groups_valid)
else:
    F_stat, p_val = np.nan, np.nan

print(f"\nOne-way ANOVA — cluster predicts post-{TEST_MIN} total conflict:")
print(f"  F = {F_stat:.3f},  p = {p_val:.4f}")
if np.isfinite(p_val):
    if p_val < 0.05:
        print("  → Significant: cluster membership predicts post-2019 conflict intensity.")
    elif p_val < 0.10:
        print("  → Marginal (p < 0.10): moderate predictive signal.")
    else:
        print("  → Not significant: clusters do not predict future conflict counts alone.")

print(f"\nPost-{TEST_MIN} mean total conflict by OOS cluster:")
oos_summary = (
    oos_val.groupby("cluster_oos")["total_conflict"]
    .agg(["mean", "median", "std", "count"])
    .round(3)
)
print(oos_summary.to_string())

# ── Save ─────────────────────────────────────────────────────────────────
oos_val.to_csv(TBL_DIR / "section4_oos_validation.csv", index=False)
pd.DataFrame({"F_stat": [F_stat], "p_val": [p_val], "n_groups": [len(groups_valid)]}).to_csv(
    TBL_DIR / "section4_anova_result.csv", index=False
)
print(f"\nSaved → {TBL_DIR}")
print("\nSection 4 complete.")

Training feature matrix (1989–2018): (157, 9)
OOS validation countries: 157
Year range (test):        2019–2024

One-way ANOVA — cluster predicts post-2019 total conflict:
  F = 18.347,  p = 0.0000
  → Significant: cluster membership predicts post-2019 conflict intensity.

Post-2019 mean total conflict by OOS cluster:
              mean  median    std  count
cluster_oos                             
0            3.810   3.667  2.262     28
1            1.292   1.167  1.430     57
2            1.574   0.417  2.041     72

Saved → D:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\tables\nb07

Section 4 complete.


## Section 5 — Centroid Inspection and Archetype Labelling

Archetype labels are assigned **post-hoc** from the centroid profiles. The three provisional labels
(Safe Hegemon, Armed Instabilizer, Defensive Deterrent) may or may not match what the data shows;
a fourth label (Inert Non-Combatant) is added for countries with near-zero capability and conflict.

Labelling heuristic (applied after inspecting centroids, not hard-coded to match the hypothesis):

| Archetype | Signature |
|---|---|
| Safe Hegemon | High MTS, below-median conflict activity, low war ratio |
| Armed Instabilizer | High MTS, above-median minor conflicts, high extraterritorial share |
| Defensive Deterrent | Low-moderate MTS, above-median conflict activity, some war participation |
| Inert Non-Combatant | Very low MTS and conflict activity |

If fewer or more than four clusters emerge, labels are assigned to the available clusters
by matching centroid signatures to the closest archetype description.

In [6]:
# ── Attach cluster labels to feature dataframe ────────────────────────────
feat_df_labeled = feat_df.copy()
feat_df_labeled["cluster"] = final_labels

# ── Centroid table (unscaled, interpretable) ─────────────────────────────
centroid_raw = (
    feat_df_labeled.groupby("cluster")[feat_cols]
    .mean()
    .round(4)
)
cluster_sizes = feat_df_labeled["cluster"].value_counts().sort_index()

print("=== Cluster centroids (unscaled mean values) ===\n")
print(centroid_raw.T.to_string())
print(f"\nCluster sizes:\n{cluster_sizes.to_string()}")

# ── Post-hoc archetype labelling ─────────────────────────────────────────
def assign_archetype(row, all_centroids):
    """
    Assign archetype label from centroid profile.
    Comparisons are relative to the median across clusters, so the function is
    scale-agnostic and works for any k.
    """
    mts      = row["mts_pca_3feat_mean"]
    minor    = row["part_n_minor_mean"]
    war      = row["part_n_war_mean"]
    activity = row.get("conflict_active_pct", 0)
    extshare = row.get("extraterritorial_share_mean", 0)

    mts_med      = all_centroids["mts_pca_3feat_mean"].median()
    minor_med    = all_centroids["part_n_minor_mean"].median()
    activity_med = all_centroids.get("conflict_active_pct", pd.Series([0.1])).median()
    extshare_med = all_centroids.get("extraterritorial_share_mean", pd.Series([0.0])).median()

    if mts <= mts_med * 0.5 and activity < activity_med * 0.5:
        return "Inert Non-Combatant"
    if mts > mts_med and extshare > extshare_med and minor > minor_med:
        return "Armed Instabilizer"
    if mts > mts_med:
        return "Safe Hegemon"
    if activity > activity_med or war > 0:
        return "Defensive Deterrent"
    return "Inert Non-Combatant"

archetype_map = {
    c: assign_archetype(centroid_raw.loc[c], centroid_raw)
    for c in range(OPTIMAL_K)
}
feat_df_labeled["archetype"] = feat_df_labeled["cluster"].map(archetype_map)

print("\n=== Archetype assignments ===")
for c in range(OPTIMAL_K):
    label    = archetype_map[c]
    n        = cluster_sizes[c]
    mts_c    = centroid_raw.loc[c, "mts_pca_3feat_mean"]
    war_c    = centroid_raw.loc[c, "part_n_war_mean"]
    minor_c  = centroid_raw.loc[c, "part_n_minor_mean"]
    stable_c = "STABLE" if mean_jacc[c] >= JACCARD_THRESHOLD else "UNSTABLE"
    print(f"  Cluster {c}: {label:25s} n={n:3d}  MTS={mts_c:.3f}  "
          f"war={war_c:.3f}  minor={minor_c:.3f}  [{stable_c}]")

print("\n=== Headline countries by cluster ===")
for c in range(OPTIMAL_K):
    members    = feat_df_labeled.loc[feat_df_labeled["cluster"] == c, "iso3"].values
    headliners = [iso3 for iso3 in HEADLINE_COUNTRIES if iso3 in members]
    top5 = (
        feat_df_labeled[feat_df_labeled["cluster"] == c]
        .nlargest(5, "mts_pca_3feat_mean")["iso3"].values
    )
    print(f"  C{c} {archetype_map[c]}:")
    print(f"    Top-5 MTS:  {', '.join(top5)}")
    if headliners:
        print(f"    Headliners: {', '.join(headliners)}")

# ── Save ─────────────────────────────────────────────────────────────────
feat_df_labeled.to_csv(TBL_DIR / "section5_cluster_assignments.csv", index=False)
centroid_raw.to_csv(TBL_DIR / "section5_centroids_raw.csv")

centroid_annotated = centroid_raw.copy()
centroid_annotated.insert(0, "archetype", [archetype_map[c] for c in centroid_annotated.index])
centroid_annotated.insert(1, "n_countries", [cluster_sizes[c] for c in centroid_raw.index])
centroid_annotated.insert(
    2, "stable",
    [mean_jacc[c] >= JACCARD_THRESHOLD for c in centroid_raw.index]
)
centroid_annotated.to_csv(TBL_DIR / "section5_centroids_annotated.csv")
print(f"\nSaved → {TBL_DIR}")
print("\nSection 5 complete.")

=== Cluster centroids (unscaled mean values) ===

cluster                           0       1       2
mts_pca_3feat_mean           0.2782  0.3142  0.1476
mts_milex_mean               0.5542  0.5847  0.3409
part_n_war_mean              0.3051  0.4158  0.0513
part_n_minor_mean            0.5648  1.8660  0.4886
war_minor_ratio_mean         0.4647  0.1879  0.1293
extraterritorial_share_mean  1.4472  0.5364  0.9088
vdem_v2x_polyarchy_mean      0.6437  0.4015  0.4450
conflict_active_pct          0.3923  0.7810  0.2235

Cluster sizes:
cluster
0    57
1    34
2    66

=== Archetype assignments ===
  Cluster 0: Defensive Deterrent       n= 57  MTS=0.278  war=0.305  minor=0.565  [UNSTABLE]
  Cluster 1: Safe Hegemon              n= 34  MTS=0.314  war=0.416  minor=1.866  [UNSTABLE]
  Cluster 2: Defensive Deterrent       n= 66  MTS=0.148  war=0.051  minor=0.489  [UNSTABLE]

=== Headline countries by cluster ===
  C0 Defensive Deterrent:
    Top-5 MTS:  SAU, JPN, ARE, AUS, DEU
    Headliners: SAU, B

## Section 6 — Visualization

Three figures:
1. **k-selection curves** — silhouette score and gap statistic vs k, with optimal k marked
2. **Cluster scatter (PCA projection)** — countries in 2D PCA space, colored by archetype, headline countries labeled
3. **Centroid heatmap** — feature means per cluster, row-normalized 0–1 for visual comparison, with raw values annotated

In [7]:
# ── Fig 1: k-selection curves ────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
k_vals        = list(K_RANGE)
sil_vals_plot = [sil_scores[k] for k in k_vals]
gap_vals_plot = [gaps[k - 1]   for k in k_vals]
gap_sks_plot  = [sks[k - 1]    for k in k_vals]

ax1.plot(k_vals, sil_vals_plot, "o-", color="steelblue", lw=2, markersize=6)
ax1.axvline(OPTIMAL_K, color="firebrick", linestyle="--", lw=1.4,
            label=f"Optimal k = {OPTIMAL_K}")
ax1.set_xlabel("Number of clusters k")
ax1.set_ylabel("Silhouette score")
ax1.set_title("Silhouette score vs k")
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

ax2.errorbar(k_vals, gap_vals_plot, yerr=gap_sks_plot, fmt="o-",
             color="seagreen", capsize=4, lw=2, markersize=6)
ax2.axvline(gap_optimal_k, color="firebrick", linestyle="--", lw=1.4,
            label=f"Gap-optimal k = {gap_optimal_k}")
ax2.set_xlabel("Number of clusters k")
ax2.set_ylabel("Gap statistic")
ax2.set_title("Gap statistic vs k  (\u00b11 SE)")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

fig.suptitle("k-Selection: Silhouette and Gap Statistic", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_k_selection.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print("Fig 1 saved.")


# ── Fig 2: PCA-reduced scatter ────────────────────────────────────────────
pca2  = PCA(n_components=2, random_state=SEED)
X_pca = pca2.fit_transform(X)
var_exp = pca2.explained_variance_ratio_

palette = sns.color_palette("tab10", OPTIMAL_K)

fig, ax = plt.subplots(figsize=(12, 7))
for c in range(OPTIMAL_K):
    mask   = final_labels == c
    label_ = f"C{c}: {archetype_map[c]}  (n={mask.sum()})"
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               color=palette[c], s=45, alpha=0.7, label=label_)

# Label headline countries
for iso3 in HEADLINE_COUNTRIES:
    idx_arr = np.where(country_labels_arr == iso3)[0]
    if len(idx_arr) > 0:
        i = idx_arr[0]
        ax.annotate(
            iso3, (X_pca[i, 0], X_pca[i, 1]),
            fontsize=7.5, ha="center", va="bottom",
            xytext=(0, 5), textcoords="offset points",
            fontweight="bold",
        )

ax.set_xlabel(f"PC1  ({var_exp[0]*100:.1f}% variance explained)")
ax.set_ylabel(f"PC2  ({var_exp[1]*100:.1f}% variance explained)")
ax.set_title(
    f"Country Conflict-Capability Archetypes  (k={OPTIMAL_K}, PCA projection)",
    fontsize=12,
)
ax.legend(fontsize=8, loc="best", framealpha=0.85)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_cluster_scatter.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print("Fig 2 saved.")


# ── Fig 3: Centroid heatmap (normalized 0–1, raw values as annotations) ─────
col_rename = {
    "mts_pca_3feat_mean":          "MTS (PCA-3feat)",
    "mts_milex_mean":               "MTS (MILEX)",
    "part_n_war_mean":              "Wars (mean annual)",
    "part_n_minor_mean":            "Minor conflicts (mean)",
    "war_minor_ratio_mean":         "War/Minor ratio",
    "extraterritorial_share_mean":  "Extraterritorial share",
    "vdem_v2x_polyarchy_mean":      "V-Dem polyarchy",
    "conflict_active_pct":          "Conflict activity (%)",
}

row_idx = [f"C{c}: {archetype_map[c]}\n(n={cluster_sizes[c]}, J={mean_jacc[c]:.2f})"
           for c in range(OPTIMAL_K)]

centroid_display = centroid_raw.copy()
centroid_display.index = row_idx
centroid_display = centroid_display.rename(columns=col_rename)

# Normalize row-wise 0–1 across clusters for each feature
centroid_norm = (centroid_display - centroid_display.min()) / (
    centroid_display.max() - centroid_display.min() + 1e-10
)

fig, ax = plt.subplots(figsize=(14, max(3.5, OPTIMAL_K * 1.1)))
sns.heatmap(
    centroid_norm,
    annot=centroid_display.round(3),
    fmt="",
    cmap="YlOrRd",
    vmin=0, vmax=1,
    ax=ax,
    linewidths=0.5,
    cbar_kws={"label": "Normalized (0 = min cluster, 1 = max cluster)"},
)
ax.set_title(
    f"Centroid profiles — k={OPTIMAL_K} archetypes  "
    f"(values = raw means; color = row-normalized)",
    fontsize=11,
)
ax.set_xlabel("")
ax.set_ylabel("")
plt.xticks(rotation=30, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_centroid_heatmap.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print("Fig 3 saved.")

print("\nSection 6 complete.")

Fig 1 saved.
Fig 2 saved.
Fig 3 saved.

Section 6 complete.


## Section 7 — Sanity Checks

In [8]:
results = []

def chk(label, expr):
    status = "PASS" if expr else "FAIL"
    results.append((label, status))
    print(f"[{status}] {label}")


# [1] Feature matrix has enough countries for meaningful clustering
n_countries_feat = len(feat_df)
chk(f"[1] Feature matrix has >= 50 countries (got {n_countries_feat})",
    n_countries_feat >= 50)

# [2] Optimal k in expected range
chk(f"[2] Optimal k in [2, 8] (got {OPTIMAL_K})",
    2 <= OPTIMAL_K <= 8)

# [3] All clusters non-trivially sized (>=3 to allow ANOVA)
min_cluster_size = int(cluster_sizes.min())
chk(f"[3] All clusters have >= 3 members (min = {min_cluster_size})",
    min_cluster_size >= 3)

# [4] Bootstrap matrix has correct shape
chk(f"[4] Bootstrap Jaccard matrix is ({N_BOOTSTRAP} × {OPTIMAL_K})",
    all_jacc.shape == (N_BOOTSTRAP, OPTIMAL_K))

# [5] At least one stable cluster
n_stable = len(stable_clusters)
chk(f"[5] At least one cluster stable (J > {JACCARD_THRESHOLD}) — {n_stable} found",
    n_stable >= 1)

# [6] OOS ANOVA ran successfully
chk(f"[6] OOS ANOVA completed (F = {F_stat:.3f}, p = {p_val:.4f})",
    np.isfinite(F_stat))

# [7] All archetype labels assigned
chk(f"[7] All {OPTIMAL_K} clusters have archetype labels",
    len(archetype_map) == OPTIMAL_K
    and all(v is not None for v in archetype_map.values()))

# [8] USA lands in a high-MTS cluster
usa_idx = np.where(country_labels_arr == "USA")[0]
if len(usa_idx) > 0:
    usa_cluster_id  = int(final_labels[usa_idx[0]])
    usa_cluster_mts = centroid_raw.loc[usa_cluster_id, "mts_pca_3feat_mean"]
    mts_rank_usa    = int(centroid_raw["mts_pca_3feat_mean"].rank(ascending=False)[usa_cluster_id])
    chk(f"[8] USA in a top-2 MTS cluster (rank={mts_rank_usa} of {OPTIMAL_K})",
        mts_rank_usa <= 2)
else:
    chk("[8] USA found in feature matrix", False)

# [9] silhouette scores are monotonically evaluated (no NaN)
chk("[9] Silhouette scores computed for all k in range",
    all(np.isfinite(sil_scores[k]) for k in K_RANGE))

# [10] All expected output files saved
expected_tables = [
    "section1_feature_matrix.csv",
    "section2_k_selection.csv",
    "section3_bootstrap_summary.csv",
    "section4_oos_validation.csv",
    "section5_cluster_assignments.csv",
]
expected_figs = [
    "fig1_k_selection.png",
    "fig2_cluster_scatter.png",
    "fig3_centroid_heatmap.png",
]
chk("[10] All 5 tables and 3 figures saved",
    all((TBL_DIR / t).exists() for t in expected_tables)
    and all((FIG_DIR / f).exists() for f in expected_figs))

n_pass = sum(1 for _, r in results if r == "PASS")
n_fail = sum(1 for _, r in results if r == "FAIL")
print(f"\n{n_pass}/{len(results)} checks passed, {n_fail} failed")

[PASS] [1] Feature matrix has >= 50 countries (got 157)
[PASS] [2] Optimal k in [2, 8] (got 3)
[PASS] [3] All clusters have >= 3 members (min = 34)
[PASS] [4] Bootstrap Jaccard matrix is (100 × 3)
[FAIL] [5] At least one cluster stable (J > 0.75) — 0 found
[PASS] [6] OOS ANOVA completed (F = 18.347, p = 0.0000)
[PASS] [7] All 3 clusters have archetype labels
[PASS] [8] USA in a top-2 MTS cluster (rank=1 of 3)
[PASS] [9] Silhouette scores computed for all k in range
[PASS] [10] All 5 tables and 3 figures saved

9/10 checks passed, 1 failed


## Section 8 — Headline Findings

In [9]:
print("=" * 70)
print("NB-07 RQ3 HEADLINE FINDINGS")
print("=" * 70)
print()
print(f"Method:     K-Means, k={OPTIMAL_K} (silhouette + gap statistic)")
print(f"Features:   {len(feat_cols)} country-level means (1989–{ANALYSIS_MAX})")
print(f"Countries:  {len(feat_df)}")
print()
print("k-Selection:")
print(f"  Silhouette optimal k = {sil_optimal_k}  "
      f"(score = {sil_scores[sil_optimal_k]:.4f})")
print(f"  Gap-statistic optimal k = {gap_optimal_k}")
print(f"  Final choice: k = {OPTIMAL_K}")
print()
print("Bootstrap stability (n_boot=100, Jaccard threshold=0.75):")
for c in range(OPTIMAL_K):
    flag = "STABLE" if mean_jacc[c] >= JACCARD_THRESHOLD else "UNSTABLE"
    print(f"  Cluster {c} ({archetype_map[c]}): "
          f"mean Jaccard = {mean_jacc[c]:.3f}  [{flag}]")
print()
print(f"Out-of-sample validation "
      f"(train 1989–{TRAIN_MAX}, test {TEST_MIN}–{ANALYSIS_MAX}):")
print(f"  ANOVA F = {F_stat:.3f},  p = {p_val:.4f}")
if np.isfinite(p_val):
    if p_val < 0.05:
        print("  → Cluster membership significantly predicts post-2019 conflict.")
    elif p_val < 0.10:
        print("  → Marginal evidence (p < 0.10) that clusters predict future conflict.")
    else:
        print("  → Clusters do not significantly predict post-2019 conflict counts.")
print()
print("Archetypes (post-hoc labelling from centroid inspection):")
for c in range(OPTIMAL_K):
    members_hl = [
        iso3 for iso3 in HEADLINE_COUNTRIES
        if iso3 in feat_df_labeled.loc[feat_df_labeled["cluster"] == c, "iso3"].values
    ]
    print(f"  Cluster {c}: {archetype_map[c]}")
    print(f"    n={cluster_sizes[c]}  "
          f"MTS={centroid_raw.loc[c, 'mts_pca_3feat_mean']:.3f}  "
          f"war={centroid_raw.loc[c, 'part_n_war_mean']:.3f}  "
          f"minor={centroid_raw.loc[c, 'part_n_minor_mean']:.3f}  "
          f"Jaccard={mean_jacc[c]:.3f}")
    if members_hl:
        print(f"    Headliners: {', '.join(members_hl)}")
print()
print(f"Outputs:")
print(f"  Tables  → {TBL_DIR}/")
print(f"  Figures → {FIG_DIR}/")
print()
print("=== NB-07 complete — proceed to NB-08 (Visualization & Summary) ===")

NB-07 RQ3 HEADLINE FINDINGS

Method:     K-Means, k=3 (silhouette + gap statistic)
Features:   8 country-level means (1989–2024)
Countries:  157

k-Selection:
  Silhouette optimal k = 3  (score = 0.2572)
  Gap-statistic optimal k = 4
  Final choice: k = 3

Bootstrap stability (n_boot=100, Jaccard threshold=0.75):
  Cluster 0 (Defensive Deterrent): mean Jaccard = 0.549  [UNSTABLE]
  Cluster 1 (Safe Hegemon): mean Jaccard = 0.533  [UNSTABLE]
  Cluster 2 (Defensive Deterrent): mean Jaccard = 0.569  [UNSTABLE]

Out-of-sample validation (train 1989–2018, test 2019–2024):
  ANOVA F = 18.347,  p = 0.0000
  → Cluster membership significantly predicts post-2019 conflict.

Archetypes (post-hoc labelling from centroid inspection):
  Cluster 0: Defensive Deterrent
    n=57  MTS=0.278  war=0.305  minor=0.565  Jaccard=0.549
    Headliners: SAU, BRA, UKR
  Cluster 1: Safe Hegemon
    n=34  MTS=0.314  war=0.416  minor=1.866  Jaccard=0.533
    Headliners: USA, RUS, CHN, IND, GBR, FRA, PAK, ISR, TUR, IR